In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-07-22 03:34:47--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.005s  

2026-07-22 03:34:47 (199 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [ ]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# All unique characters present in the dataset
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Bidirectional mapping lookup tables
stoi = { ch:i for i, ch in enumerate(chars) }
itos = { i:ch for i, ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [ ]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 64  # Independent sequences per batch
block_size = 256 # Context length (T)

def get_batch(split):
    dataset = train_data if split == 'train' else val_data
    # Grab random starting index offsets
    ix = torch.randint(len(dataset) - block_size, (batch_size,))

    x = torch.stack([dataset[i : i + block_size] for i in ix])
    y = torch.stack([dataset[i + 1 : i + block_size + 1] for i in ix])

    return x.to(device), y.to(device)

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class Head(nn.Module):
    def __init__(self, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Lower-triangular causal mask buffer (not a learnable parameter)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        v = self.value(x) # (B, T, head_size)

        # Compute raw attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5) # (B, T, T)

        # Causal masking: prevent token t from looking at t+1...T
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)

        # Weighted aggregation of values
        out = wei @ v # (B, T, head_size)
        return out


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([
            Head(head_size, n_embd, block_size, dropout) for _ in range(num_heads)
        ])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Concatenate outputs along channel dimension: (B, T, num_heads * head_size)
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embd, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Pre-LayerNorm formulation with skip connections
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size, n_head, n_layer, dropout):
        super().__init__()
        self.block_size = block_size

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx) # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T, n_embd)

        x = tok_emb + pos_emb # Broadcasting addition -> (B, T, n_embd)
        x = self.blocks(x)    # Pass through stacked Transformer blocks
        x = self.ln_f(x)      # Final norm
        logits = self.lm_head(x) # Map to vocabulary space -> (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            # Reshape for PyTorch cross_entropy format: (N, C)
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # Truncate context if sequence exceeds block size
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)

            # Extract logits of the last time step
            logits = logits[:, -1, :] # (B, C)
            probs = F.softmax(logits, dim=-1) # Convert to probabilities

            idx_next = torch.multinomial(probs, num_samples=1) # Sample next token -> (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # Append to sequence -> (B, T+1)
        return idx

In [ ]:
# --- Model Config ---
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
eval_iters = 200

model = GPTLanguageModel(vocab_size, n_embd, block_size, n_head, n_layer, dropout).to(device)
print(f"Total Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

Total Parameters: 10.79M


In [ ]:
learning_rate = 3e-4
max_iters = 5000
eval_interval = 500

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iteration in range(max_iters):
    # Periodically compute validation loss
    if iteration % eval_interval == 0 or iteration == max_iters - 1:
        losses = estimate_loss()
        print(f"Iter {iteration:4d} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

Iter    0 | Train Loss: 4.3198 | Val Loss: 4.3210
Iter  500 | Train Loss: 1.9025 | Val Loss: 2.0083
Iter 1000 | Train Loss: 1.5438 | Val Loss: 1.7200
Iter 1500 | Train Loss: 1.4035 | Val Loss: 1.6119
Iter 2000 | Train Loss: 1.3195 | Val Loss: 1.5515
Iter 2500 | Train Loss: 1.2571 | Val Loss: 1.5207
Iter 3000 | Train Loss: 1.2117 | Val Loss: 1.4973
Iter 3500 | Train Loss: 1.1712 | Val Loss: 1.4933
Iter 4000 | Train Loss: 1.1328 | Val Loss: 1.4924
Iter 4500 | Train Loss: 1.1017 | Val Loss: 1.4857
Iter 4999 | Train Loss: 1.0684 | Val Loss: 1.4919


In [ ]:
# Seed context with a single newline character index (0)
context = torch.zeros((1, 1), dtype=torch.long, device=device)

# Generate 500 tokens
generated_indices = model.generate(context, max_new_tokens=500)[0].tolist()
print("\n--- Model Output Sample ---")
print(decode(generated_indices))
print("---------------------------\n")

# Save parameter weights to disk
torch.save(model.state_dict(), 'nanogpt_weights.pth')
print("Model weights exported to nanogpt_weights.pth")


--- Model Output Sample ---

If I and Tybalt to-morrow; I will releft on the other
suffen and save to touch the contrary.

FLORIZEL:
I will comfort it to pies his life in toad
word: patient; the citizens good consently.

RICHMOND:
A rarier that I could I conjust
I know thyself.

LUCIO:
Concusicial, live here am I, come townd away.

LUCIO:
Certainly! How make her was hence?

DERBELET:
Petruchio, just; I will I pray up Henry.

DUKE VINCENTIO:
I am the not value a trick of minded
fast: therefore, beg 'em four stards of English
---------------------------

Model weights exported to nanogpt_weights.pth
